In [8]:
!pip install pandas scikit-learn

import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

## Q1-Q2: Target variable and business alignment

Rather than predicting `hired` (which only mimics past recruiter decisions
and their biases), I define a derived target based on true outcomes:

**sales_to_salary_ratio = sales_achieved_after_one_year_gbp / expected_salary_gbp**

This aligns with the business objective: minimise cost (salary) and
maximise outcome (sales) — i.e., value-for-money per hire.

In [9]:
df = pd.read_csv("sales_recruitment_synthetic_200.csv")
print("Full dataset:", df.shape)

# Keep hired candidates only — outcomes exist only for them (Q4)
df = df[df["hired"] == "Yes"].copy()
print("Hired only:", df.shape)

# Create the derived target (Q1)
df["sales_to_salary_ratio"] = (
    df["sales_achieved_after_one_year_gbp"] / df["expected_salary_gbp"]
)
df["sales_to_salary_ratio"].describe()

Full dataset: (200, 14)
Hired only: (112, 14)


count    112.000000
mean       2.449301
std        0.628671
min        1.176596
25%        2.007528
50%        2.359452
75%        2.813844
max        4.124324
Name: sales_to_salary_ratio, dtype: float64

## Q3: Regression, not classification

The target is a continuous quantity (ratio from ~1.2 to ~4.1), so this is
a **regression** problem (Y quantitative → regression; Y finite set →
classification). Evaluation uses MSE/RMSE, not accuracy.

In [10]:
# Features = interview-time information ONLY.
# Excluded to prevent leakage: sales, manager satisfaction, retention, hired
# (these are only known AFTER hiring)
features = df[[
    "age", "years_experience", "highest_education",
    "technical_test_score", "interview_rating",
    "communication_assessment", "leadership_assessment",
    "expected_salary_gbp", "previous_job_tenure_years", "gender"
]]

# One-hot encode categorical features
X = pd.get_dummies(features, drop_first=True)
y = df["sales_to_salary_ratio"]

# 80/20 train/test split — evaluate on unseen data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (KNN is distance-based)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set:", X_train.shape, " Test set:", X_test.shape)

Training set: (89, 13)  Test set: (23, 13)


In [11]:
# Naive baseline: always predict the training-set mean
baseline_rmse = ((y_test - y_train.mean())**2).mean()**0.5
print("Baseline RMSE (predict mean):", round(baseline_rmse, 3))

# KNN regressor
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_rmse = mean_squared_error(y_test, knn.predict(X_test_scaled)) ** 0.5
print("KNN RMSE:", round(knn_rmse, 3))

# Linear regression (comparison)
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_rmse = mean_squared_error(y_test, lr.predict(X_test)) ** 0.5
print("Linear Regression RMSE:", round(lr_rmse, 3))

Baseline RMSE (predict mean): 0.633
KNN RMSE: 0.574
Linear Regression RMSE: 0.602


In [12]:
# 5-fold cross-validation for a more stable error estimate
cv_scores = cross_val_score(
    KNeighborsRegressor(n_neighbors=5),
    scaler.fit_transform(X), y,
    cv=5, scoring="neg_root_mean_squared_error"
)
print("5-fold CV RMSE:", round(-cv_scores.mean(), 3),
      "+/-", round(cv_scores.std(), 3))

5-fold CV RMSE: 0.616 +/- 0.082


## Q4: Preprocessing summary & limitations

- Filtered to hired candidates (data necessity — outcomes only exist for them)
- Created derived ratio target; removed post-hire leakage columns
- One-hot encoded categoricals; scaled features for KNN; 80/20 split + 5-fold CV
- **Limitation:** selection bias — we only observe outcomes for candidates
  the old process hired, so predictions for historically-rejected profiles
  are less reliable
- **Ethics:** gender/age used here for demonstration; in production these
  should be excluded and outcomes audited for disparate impact